# Section 06 Onboarding Notebook: Classical/Ab-Initio Molecular Dynamics & Tully Fewest-Switches Surface Hopping (FSSH)

### Overview & Pedagogical Design
When nuclear motion is confined to a single well-separated electronic state, nuclei evolve classically on a Born–Oppenheimer Potential Energy Surface (PES) via symplectic **Velocity Verlet** integration ([Velocity Verlet BOMD](https://docs.matlantis.com/atomistic-simulation-tutorial/en/6_1_md-nve.html)). However, in photochemistry, conical intersections, and charge transfer, electronic states become nearly degenerate and the Born–Oppenheimer approximation breaks down. **Tully's Fewest-Switches Surface Hopping (FSSH)** ([Tully FSSH Pedagogical Overview](https://pubs.acs.org/acsodf/article/7/50/45810/461699/Pedagogical-Overview-of-the-Fewest-Switches)) evolves an ensemble of classical nuclear trajectories on active adiabatic surfaces $E_k(R)$ while integrating the coherent electronic amplitudes $c_k(t)$ along each trajectory and stochastically hopping between surfaces with the minimum switching probability required to match quantum populations $|c_k(t)|^2$.

In this notebook, **no MD or FSSH functions are pre-solved for you**. You will implement **14 granular `# TODO` functions** covering every step of both algorithms:
- **Part 1 — Ground-State Molecular Dynamics: Morse Potential, Symplectic Velocity Verlet, Langevin NVT & Vibrational Spectrum (`TODO 1` – `TODO 5`)**
- **Part 2 — Nonadiabatic Representation: Tully Model I Diabatic-to-Adiabatic Diagonalization, Hellmann–Feynman Forces & NACVs (`TODO 6` – `TODO 8`)**
- **Part 3 — Tully Fewest-Switches Surface Hopping: Electronic RK4 Sub-Stepping, Hop Probabilities, Velocity Rescaling & Decoherence (`TODO 9` – `TODO 14`)**
- **Part 4 — Step-by-Step Checkpoint & Verification Suite (`run_section06_checkpoints`)**

In [ ]:
import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)
print("Section 06 environment initialized. Implement TODOs 1-14 and run `run_section06_checkpoints()`!")

## Part 1: Ground-State Molecular Dynamics — Morse Potential, Velocity Verlet, Langevin NVT & Vibrational Spectrum (TODOs 1–5)

### Mathematical Specifications
1. **Morse Potential & Analytical Force (`morse_potential_and_force` — TODO 1)**:
   For a diatomic bond with dissociation energy $D_e$, width parameter $a$, and equilibrium bond length $r_{\text{eq}}$:
   $$V(r) = D_e\left(1 - e^{-a(r - r_{\text{eq}})}\right)^2, \qquad F(r) = -\frac{dV}{dr} = -2 a D_e\left(1 - e^{-a(r - r_{\text{eq}})}\right)e^{-a(r - r_{\text{eq}})}$$

2. **Single Symplectic Velocity Verlet Step (`velocity_verlet_step` — TODO 2)** ([Velocity Verlet BOMD](https://docs.matlantis.com/atomistic-simulation-tutorial/en/6_1_md-nve.html)):
   $$v\!\left(t + \tfrac{\Delta t}{2}\right) = v(t) + \frac{F(r(t))}{2m}\Delta t, \qquad r(t + \Delta t) = r(t) + v\!\left(t + \tfrac{\Delta t}{2}\right)\Delta t, \qquad v(t + \Delta t) = v\!\left(t + \tfrac{\Delta t}{2}\right) + \frac{F(r(t + \Delta t))}{2m}\Delta t$$

3. **NVE Trajectory Driver & Symplectic Shadow-Hamiltonian Check (`run_nve_morse_trajectory` — TODO 3)**:
   Propagate $(r(t), v(t))$ for `n_steps` steps and record $E_{\text{kin}}(t) = \frac{1}{2}mv(t)^2$, $E_{\text{pot}}(t) = V(r(t))$, $E_{\text{tot}}(t)$, and maximum energy drift $\max_t |E_{\text{tot}}(t) - E_{\text{tot}}(0)|$.

4. **BAOAB / Impulse Langevin NVT Thermostat Step (`langevin_nvt_step` — TODO 4)**:
   To sample a canonical NVT ensemble at temperature $k_B T$ with friction coefficient $\gamma$, apply an Ornstein–Uhlenbeck stochastic velocity-rescaling step around the Velocity Verlet step:
   $$c_1 = e^{-\gamma \Delta t}, \qquad c_2 = \sqrt{(1 - c_1^2)\frac{k_B T}{m}}, \qquad v \leftarrow c_1 v + c_2\,\xi \quad (\xi \sim \mathcal{N}(0, 1))$$

5. **Velocity Autocorrelation Function (VACF) & Vibrational Power Spectrum (`compute_vacf_and_power_spectrum` — TODO 5)**:
   Compute the normalized velocity autocorrelation function $C_{vv}(t) = \frac{\langle v(0)v(t)\rangle}{\langle v^2\rangle}$ and its cosine Fourier transform $I(\omega) = \int_0^{T_{\max}} C_{vv}(t)\cos(\omega t)\,e^{-t/\tau_{\text{win}}}\,dt$ to extract the anharmonic vibrational frequency $\omega_{\text{anharm}}$.

In [ ]:
def morse_potential_and_force(
    r: float, d_e: float = 0.1745, a: float = 1.0282, r_eq: float = 1.4011
) -> tuple[float, float]:
    '''
    TODO 1: Evaluate the Morse potential V(r) and analytical force F(r) = -dV/dr in atomic units.
    Return (V, F).
    '''
    # TODO 1: Compute exp_term = np.exp(-a * (r - r_eq)) and return (V, F).
    raise NotImplementedError("Complete TODO 1: morse_potential_and_force")


def velocity_verlet_step(
    r: float, v: float, f: float, mass: float, dt: float, force_fn
) -> tuple[float, float, float, float]:
    '''
    TODO 2: Advance (r, v) by one symplectic Velocity Verlet step of size `dt` using `force_fn(r_new) -> (v_pot_new, f_new)`.
    Return (r_new, v_new, v_pot_new, f_new).
    '''
    # TODO 2: Implement half-step velocity update, full-step position update, new force evaluation, and second half-step velocity update.
    raise NotImplementedError("Complete TODO 2: velocity_verlet_step")


def run_nve_morse_trajectory(
    r0: float = 1.80, v0: float = 0.0, mass: float = 918.57, dt: float = 2.0, n_steps: int = 500
) -> dict[str, np.ndarray | float]:
    '''
    TODO 3: Propagate an NVE Morse oscillator trajectory for `n_steps` steps using `velocity_verlet_step`.
    Return a dict with keys 'times', 'positions', 'velocities', 'kinetic_energy',
    'potential_energy', 'total_energy', and 'max_energy_drift'.
    '''
    # TODO 3: Allocate trajectory arrays, loop over n_steps calling velocity_verlet_step, and compute max_energy_drift.
    raise NotImplementedError("Complete TODO 3: run_nve_morse_trajectory")


def langevin_nvt_step(
    r: float,
    v: float,
    f: float,
    mass: float,
    dt: float,
    gamma_friction: float,
    temperature_kt: float,
    rng: np.random.Generator,
    force_fn,
) -> tuple[float, float, float, float]:
    '''
    TODO 4: Perform one BAOAB-style Langevin NVT step (Velocity Verlet + Ornstein-Uhlenbeck thermostat).
    Return (r_new, v_new, v_pot_new, f_new).
    '''
    # TODO 4: Apply half-step thermostat (dt/2), Velocity Verlet step (dt), and second half-step thermostat (dt/2).
    raise NotImplementedError("Complete TODO 4: langevin_nvt_step")


def compute_vacf_and_power_spectrum(
    velocities: np.ndarray, dt: float
) -> tuple[np.ndarray, np.ndarray, float]:
    '''
    TODO 5: Compute the normalized velocity autocorrelation function C_vv(t), the frequency grid
    omega_au (via rfftfreq), and the peak vibrational angular frequency omega_peak_au.
    Return (vacf, omega_grid_au, omega_peak_au).
    '''
    # TODO 5: Subtract mean velocity, compute autocorrelation via np.correlate or FFT, normalize by vacf[0],
    #         compute power spectrum |rfft(vacf * hanning_window)|, and locate omega_peak_au.
    raise NotImplementedError("Complete TODO 5: compute_vacf_and_power_spectrum")

## Part 2: Nonadiabatic Representation — Tully Model I Avoided Crossing, Adiabatic Forces & NACVs (TODOs 6–8)

### Mathematical Specifications
1. **Tully Model I Diabatic Hamiltonian & Analytical Derivative (`tully_model1_diabatic_and_grad` — TODO 6)** ([Tully 1990 / ACS Omega Review](https://pubs.acs.org/acsodf/article/7/50/45810/461699/Pedagogical-Overview-of-the-Fewest-Switches)):
   $$V_{11}(R) = \begin{cases} A\left(1 - e^{-B R}\right) & R \ge 0 \\ -A\left(1 - e^{B R}\right) & R < 0 \end{cases}, \qquad V_{22}(R) = -V_{11}(R), \qquad V_{12}(R) = V_{21}(R) = C\,e^{-D R^2}$$
   $$\frac{dV_{11}}{dR} = A B\,e^{-B|R|}, \qquad \frac{dV_{22}}{dR} = -\frac{dV_{11}}{dR}, \qquad \frac{dV_{12}}{dR} = -2 C D R\,e^{-D R^2}$$

2. **Adiabatic Eigenstates, Hellmann–Feynman Forces & Nonadiabatic Coupling $d_{01}(R)$ (`compute_adiabatic_surfaces_and_nacv` — TODO 7)**:
   Diagonalize the $2 \times 2$ diabatic matrix $\mathbf{V}_{\text{diab}}(R)$ using the continuous mixing angle $\theta(R) = \frac{1}{2}\arctan\!\left(\frac{2 V_{12}}{V_{22} - V_{11}}\right)$ (`0.5 * np.arctan2(2 * V12, V22 - V11)`):
   $$\mathbf{U}(R) = \begin{pmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{pmatrix}, \qquad E_{0,1}(R) = \frac{V_{11}+V_{22}}{2} \mp \sqrt{\left(\frac{V_{11}-V_{22}}{2}\right)^2 + V_{12}^2}$$
   Transform the diabatic gradient matrix $\mathbf{G}^{\text{ad}}(R) = \mathbf{U}(R)^\mathsf{T}\frac{d\mathbf{V}_{\text{diab}}}{dR}\mathbf{U}(R)$ to extract:
   - **Adiabatic Hellmann–Feynman forces**: $F_k(R) = -G_{kk}^{\text{ad}}(R)$ for $k \in \{0, 1\}$.
   - **First-derivative Nonadiabatic Coupling (NAC)**:
     $$d_{01}(R) = \langle \Phi_0(R) | \nabla_R \Phi_1(R) \rangle = \frac{G_{01}^{\text{ad}}(R)}{E_1(R) - E_0(R)} = -\frac{d\theta}{dR}$$

3. **Analytical vs. Finite-Difference NAC Verification (`verify_nacv_finite_difference` — TODO 8)**:
   Verify that $\langle \Phi_0(R) | \frac{\Phi_1(R + \delta R) - \Phi_1(R - \delta R)}{2\delta R} \rangle$ matches $d_{01}(R)$ to $\mathcal{O}(\delta R^2)$ and check that $\int_{-\infty}^{+\infty} d_{01}(R)\,dR = -\frac{\pi}{2}$ (complete diabatic state inversion across the avoided crossing!).

In [ ]:
def tully_model1_diabatic_and_grad(
    r: float, A: float = 0.01, B: float = 1.6, C: float = 0.005, D: float = 1.0
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 6: Construct the 2x2 diabatic potential matrix V_diab(r) and its derivative matrix dV_diab_dr(r)
    for Tully's Simple Avoided Crossing (Model I). Return (V_diab, dV_diab_dr).
    '''
    # TODO 6: Evaluate V11, V22, V12 and their analytical derivatives dV11, dV22, dV12.
    raise NotImplementedError("Complete TODO 6: tully_model1_diabatic_and_grad")


def compute_adiabatic_surfaces_and_nacv(
    r: float, A: float = 0.01, B: float = 1.6, C: float = 0.005, D: float = 1.0
) -> tuple[np.ndarray, np.ndarray, float, np.ndarray]:
    '''
    TODO 7: Diagonalize Tully Model I at position `r` with smooth phase fixing (using mixing angle theta)
    and return (energies_ad, forces_ad, d01_nac, U_eigvecs).
    '''
    # TODO 7: Call tully_model1_diabatic_and_grad(r), compute theta = 0.5 * arctan2(2*V12, V22 - V11),
    #         form U, transform G_ad = U.T @ dV_diab @ U, and return (energies, forces, d01, U).
    raise NotImplementedError("Complete TODO 7: compute_adiabatic_surfaces_and_nacv")


def verify_nacv_finite_difference(r: float, dr: float = 1e-6) -> tuple[float, float]:
    '''
    TODO 8: Compute the numerical NACV <Phi_0(r) | (Phi_1(r+dr) - Phi_1(r-dr)) / (2*dr)>
    and return (d01_analytical, d01_numerical).
    '''
    # TODO 8: Evaluate compute_adiabatic_surfaces_and_nacv at r, r + dr, and r - dr, and return (d01, d01_fd).
    raise NotImplementedError("Complete TODO 8: verify_nacv_finite_difference")

## Part 3: Tully Fewest-Switches Surface Hopping — Electronic Sub-Stepping, Hopping, Momentum Rescaling & Decoherence (TODOs 9–14)

### Mathematical Specifications
1. **Electronic Schrödinger Equation Integration (`rk4_electronic_amplitudes_step` — TODO 9)**:
   Along a nuclear trajectory moving with velocity $v = \dot{R}$, the adiabatic electronic amplitudes $\mathbf{c}(t) = [c_0(t), c_1(t)]^\mathsf{T}$ obey:
   $$\begin{pmatrix} \dot{c}_0 \\ \dot{c}_1 \end{pmatrix} = \begin{pmatrix} -i E_0(R) & -v\,d_{01}(R) \\ +v\,d_{01}(R) & -i E_1(R) \end{pmatrix} \begin{pmatrix} c_0 \\ c_1 \end{pmatrix}$$
   Integrate this ODE over nuclear step $\Delta t$ using `n_substeps` fourth-order Runge–Kutta (RK4) sub-steps of size $\delta t = \Delta t / n_{\text{substeps}}$.

2. **Tully Fewest-Switches Hop Probability (`compute_fssh_hop_probability` — TODO 10)**:
   When the active surface is $k \in \{0, 1\}$ and the other surface is $j = 1 - k$, the population flux from $k$ to $j$ over $\Delta t$ determines the switching probability:
   $$b_{jk} = -2\,\text{Re}\!\left(c_j^* c_k \, v\, d_{jk}\right) \quad (\text{where } d_{10} = -d_{01}), \qquad g_{k \to j} = \max\!\left(0,\ \frac{b_{jk}\,\Delta t}{|c_k|^2}\right)$$

3. **Kinetic Energy Check, Velocity Rescaling & Frustrated Hops (`attempt_surface_hop_and_rescale_velocity` — TODO 11)**:
   Draw a uniform random number $\zeta \in [0, 1)$. If $\zeta < g_{k \to j}$, attempt a hop $k \to j$:
   - Potential energy change: $\Delta E_{\text{pot}} = E_j(R) - E_k(R)$.
   - If the nuclear kinetic energy $E_{\text{kin}} = \frac{1}{2}mv^2 \ge \Delta E_{\text{pot}}$, the hop **succeeds**: set `active_state = j` and rescale velocity along the NAC direction (preserving $\text{sign}(v)$ in 1D):
     $$v_{\text{new}} = \text{sign}(v)\sqrt{v^2 - \frac{2\Delta E_{\text{pot}}}{m}}$$
   - If $E_{\text{kin}} < \Delta E_{\text{pot}}$, the hop is **energetically forbidden (frustrated hop)**: keep `active_state = k` and $v_{\text{new}} = v$.

4. **Granucci–Persico Energy-Based Decoherence Correction (`apply_granucci_persico_decoherence` — TODO 12)**:
   Because electronic amplitudes $\mathbf{c}(t)$ in raw FSSH propagate coherently along a single nuclear path even after wavepackets separate on different surfaces, damp the inactive amplitude $c_j$ ($j \ne k$) at each step with characteristic decoherence time $\tau_{jk}$ ($C_{\text{dec}} = 0.1\ E_{\text{h}}$):
   $$\tau_{jk} = \frac{1}{|E_j - E_k|}\left(1 + \frac{C_{\text{dec}}}{E_{\text{kin}}}\right), \qquad c_j' = c_j\,e^{-\Delta t / \tau_{jk}}, \qquad c_k' = c_k\sqrt{\frac{1 - |c_j'|^2}{|c_k|^2}}$$

5. **Single FSSH Trajectory (`run_single_fssh_trajectory` — TODO 13)** & **Swarm Branching Ratios (`run_fssh_ensemble` — TODO 14)**:
   Combine TODOs 7–12 into a full trajectory driver and run an ensemble of trajectories to compute transmission and reflection probabilities on the lower and upper adiabatic surfaces!

In [ ]:
def rk4_electronic_amplitudes_step(
    c: np.ndarray, energies: np.ndarray, v_dot_d01: float, dt: float, n_substeps: int = 10
) -> np.ndarray:
    '''
    TODO 9: Advance the 2-component complex electronic statevector `c` over nuclear time step `dt`
    using `n_substeps` steps of 4th-order Runge-Kutta.
    '''
    # TODO 9: Define dc/dt = [-i*E0*c0 - v_dot_d01*c1, -i*E1*c1 + v_dot_d01*c0] and integrate via RK4.
    raise NotImplementedError("Complete TODO 9: rk4_electronic_amplitudes_step")


def compute_fssh_hop_probability(
    c: np.ndarray, active_state: int, v_dot_d01: float, dt: float
) -> float:
    '''
    TODO 10: Compute Tully's fewest-switches hopping probability g_{k -> j} in [0, 1]
    from `active_state` k to `other_state` j = 1 - k over step `dt`.
    '''
    # TODO 10: Set d_jk = +v_dot_d01 if (j==0 and k==1) else -v_dot_d01, evaluate b_jk = -2 * Re(c_j^* c_k d_jk),
    #          and return clip(dt * b_jk / |c_k|^2, 0.0, 1.0).
    raise NotImplementedError("Complete TODO 10: compute_fssh_hop_probability")


def attempt_surface_hop_and_rescale_velocity(
    active_state: int,
    v: float,
    mass: float,
    energies: np.ndarray,
    hop_prob: float,
    rand_u: float,
) -> tuple[int, float, bool, bool]:
    '''
    TODO 11: Given uniform random number `rand_u` in [0, 1), decide whether a surface hop occurs.
    Return (new_active_state, new_v, hopped_flag, frustrated_hop_flag).
    '''
    # TODO 11: If rand_u < hop_prob, check if 0.5 * mass * v^2 >= E_target - E_active.
    #          If sufficient, rescale v to conserve total energy and hop; otherwise record a frustrated hop.
    raise NotImplementedError("Complete TODO 11: attempt_surface_hop_and_rescale_velocity")


def apply_granucci_persico_decoherence(
    c: np.ndarray,
    active_state: int,
    energies: np.ndarray,
    e_kin: float,
    dt: float,
    c_param: float = 0.1,
) -> np.ndarray:
    '''
    TODO 12: Apply Granucci-Persico energy-based decoherence damping to inactive state j = 1 - active_state
    and renormalize active state amplitude c[active_state] so ||c||_2 == 1.
    '''
    # TODO 12: Compute tau = (1 + c_param / e_kin) / |E_1 - E_0|, damp c[j] *= exp(-dt / tau), and rescale c[k].
    raise NotImplementedError("Complete TODO 12: apply_granucci_persico_decoherence")


def run_single_fssh_trajectory(
    k0: float = 20.0,
    r0: float = -8.0,
    mass: float = 2000.0,
    dt: float = 1.0,
    max_steps: int = 6000,
    seed: int = 42,
    apply_decoherence: bool = True,
) -> dict[str, float | int | bool]:
    '''
    TODO 13: Propagate a single Tully FSSH trajectory starting at r0 on state 0 with momentum k0.
    Terminate when |r| > 10.0 Bohr or at max_steps.
    Return a dict with keys:
        'final_r', 'final_p', 'active_state', 'transmitted', 'n_hops',
        'n_frustrated_hops', 'max_energy_drift', 'norm_drift'.
    '''
    # TODO 13: Combine Velocity Verlet on the active surface with TODOs 7, 9, 10, 11, and 12.
    raise NotImplementedError("Complete TODO 13: run_single_fssh_trajectory")


def run_fssh_ensemble(
    k0: float = 20.0, n_traj: int = 40, base_seed: int = 100
) -> dict[str, float]:
    '''
    TODO 14: Run an ensemble of `n_traj` FSSH trajectories at initial momentum `k0` and return
    branching fractions: 'trans_state0', 'trans_state1', 'refl_state0', 'refl_state1', 'mean_energy_drift'.
    '''
    # TODO 14: Loop over i in range(n_traj) calling run_single_fssh_trajectory(..., seed=base_seed + i)
    #          and compute the 4 scattering channel probabilities (which must sum to 1.0).
    raise NotImplementedError("Complete TODO 14: run_fssh_ensemble")

## Part 4: Step-by-Step Checkpoint & Verification Suite

Run `run_section06_checkpoints()` below as you implement each `# TODO` function to verify your classical MD and nonadiabatic Tully FSSH solvers!

In [2]:
def run_section06_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 14 #TODO functions in Section 06 (MD + FSSH).'''
    passed = 0
    total_todos = 14

    try:
        v_eq, f_eq = morse_potential_and_force(1.4011)
        v_st, f_st = morse_potential_and_force(1.80)
        assert abs(v_eq) < 1e-15 and abs(f_eq) < 1e-15 and v_st > 0.0 and f_st < 0.0
        passed += 1
        print("[PASS] TODO 1 : morse_potential_and_force")

        r1, v1, vp1, f1 = velocity_verlet_step(1.80, 0.0, f_st, 918.57, 2.0, morse_potential_and_force)
        assert r1 < 1.80 and v1 < 0.0
        passed += 1
        print("[PASS] TODO 2 : velocity_verlet_step")

        traj_nve = run_nve_morse_trajectory(r0=1.80, v0=0.0, mass=918.57, dt=2.0, n_steps=500)
        assert traj_nve["max_energy_drift"] < 1e-5
        passed += 1
        print(f"[PASS] TODO 3 : run_nve_morse_trajectory (Max NVE drift = {traj_nve['max_energy_drift']:.2e} Eh)")

        rng_test = np.random.default_rng(7)
        r_l, v_l, vp_l, f_l = langevin_nvt_step(1.50, 0.0, 0.0, 918.57, 2.0, 0.005, 0.001, rng_test, morse_potential_and_force)
        assert isinstance(r_l, float) and isinstance(v_l, float)
        passed += 1
        print("[PASS] TODO 4 : langevin_nvt_step")

        vacf, omega_grid, omega_peak = compute_vacf_and_power_spectrum(traj_nve["velocities"], dt=2.0)
        assert np.isclose(vacf[0], 1.0, atol=1e-12) and 0.010 < omega_peak < 0.025
        passed += 1
        print(f"[PASS] TODO 5 : compute_vacf_and_power_spectrum (omega_peak = {omega_peak:.5f} a.u.)")

        V_d0, dV_d0 = tully_model1_diabatic_and_grad(0.0)
        assert V_d0.shape == (2, 2) and np.isclose(V_d0[0, 1], 0.005) and np.isclose(dV_d0[0, 0], 0.016)
        passed += 1
        print("[PASS] TODO 6 : tully_model1_diabatic_and_grad")

        e_ad, f_ad, d01_0, U_0 = compute_adiabatic_surfaces_and_nacv(0.0)
        assert np.isclose(e_ad[1] - e_ad[0], 0.010, atol=1e-12) and abs(d01_0) > 1.0
        passed += 1
        print(f"[PASS] TODO 7 : compute_adiabatic_surfaces_and_nacv (Peak |d_01(0)| = {abs(d01_0):.4f} Bohr^-1)")

        d01_an, d01_fd = verify_nacv_finite_difference(0.25, dr=1e-6)
        assert np.isclose(d01_an, d01_fd, atol=1e-7)
        passed += 1
        print(f"[PASS] TODO 8 : verify_nacv_finite_difference (diff = {abs(d01_an - d01_fd):.2e})")

        c_init = np.array([1.0 + 0.0j, 0.0 + 0.0j])
        c_step = rk4_electronic_amplitudes_step(c_init, e_ad, v_dot_d01=0.02, dt=1.0, n_substeps=10)
        assert np.isclose(np.linalg.norm(c_step), 1.0, atol=1e-12) and abs(c_step[1]) > 0.0
        passed += 1
        print("[PASS] TODO 9 : rk4_electronic_amplitudes_step")

        g_prob = compute_fssh_hop_probability(c_step, active_state=0, v_dot_d01=0.02, dt=1.0)
        assert 0.0 <= g_prob <= 1.0
        passed += 1
        print(f"[PASS] TODO 10: compute_fssh_hop_probability (g_0->1 = {g_prob:.6f})")

        st_ok, v_ok, hopped_ok, frust_ok = attempt_surface_hop_and_rescale_velocity(0, 0.02, 2000.0, e_ad, 0.5, 0.1)
        st_fr, v_fr, hopped_fr, frust_fr = attempt_surface_hop_and_rescale_velocity(0, 0.001, 2000.0, e_ad, 0.5, 0.1)
        assert st_ok == 1 and hopped_ok and not frust_ok
        assert st_fr == 0 and not hopped_fr and frust_fr
        passed += 1
        print("[PASS] TODO 11: attempt_surface_hop_and_rescale_velocity (allowed + frustrated hops verified)")

        c_dec = apply_granucci_persico_decoherence(np.array([0.8 + 0.0j, 0.6 + 0.0j]), 0, e_ad, 0.05, 1.0)
        assert np.isclose(np.linalg.norm(c_dec), 1.0, atol=1e-12) and abs(c_dec[1]) < 0.6
        passed += 1
        print("[PASS] TODO 12: apply_granucci_persico_decoherence")

        fssh_1 = run_single_fssh_trajectory(k0=20.0, seed=42)
        assert fssh_1["transmitted"] and fssh_1["max_energy_drift"] < 1e-4 and fssh_1["norm_drift"] < 1e-6
        passed += 1
        print(f"[PASS] TODO 13: run_single_fssh_trajectory (Final state = {fssh_1['active_state']}, drift = {fssh_1['max_energy_drift']:.2e} Eh)")

        ens = run_fssh_ensemble(k0=20.0, n_traj=20, base_seed=100)
        total_prob = ens["trans_state0"] + ens["trans_state1"] + ens["refl_state0"] + ens["refl_state1"]
        assert np.isclose(total_prob, 1.0, atol=1e-12) and ens["trans_state1"] > 0.25
        passed += 1
        print(f"[PASS] TODO 14: run_fssh_ensemble (T_0 = {ens['trans_state0']:.2f}, T_1 = {ens['trans_state1']:.2f})")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 06 (MD + FSSH) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section06_checkpoints()
print("Section 06 checkpoint suite loaded. Implement TODOs 1-14 and call `run_section06_checkpoints()`!")

Section 06 checkpoint suite loaded. Implement TODOs 1-14 and call `run_section06_checkpoints()`!
